In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()


,id,sender,subject,body,priority,triage_label
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond


In [3]:
def email_assistant(email_text):
    text = str(email_text).lower()

    if any(w in text for w in ["urgent", "asap", "deadline", "submit", "eod"]):
        return "notify", "urgent", 0.9

    elif any(w in text for w in ["please", "can you", "review", "confirm"]):
        return "respond", "polite", 0.8

    elif any(w in text for w in ["thank you", "thanks"]):
        return "ignore", "polite", 0.85

    else:
        return "review", "neutral", 0.4


In [4]:
results = []

for _, row in df.iterrows():
    intent, tone, confidence = email_assistant(row["body"])
    results.append({
        "id": row["id"],
        "predicted_intent": intent,
        "predicted_tone": tone,
        "confidence": confidence
    })

pred_df = pd.DataFrame(results)
pred_df.head()


,id,predicted_intent,predicted_tone,confidence
0,1,review,neutral,0.4
1,2,respond,polite,0.8
2,3,review,neutral,0.4
3,4,respond,polite,0.8
4,5,respond,polite,0.8


In [5]:
def human_in_loop(row):
    if row["confidence"] < 0.7:
        return "HUMAN_REVIEW_REQUIRED"
    else:
        return "AUTO_APPROVED"


In [6]:
pred_df["decision"] = pred_df.apply(human_in_loop, axis=1)
pred_df.head()


,id,predicted_intent,predicted_tone,confidence,decision
0,1,review,neutral,0.4,HUMAN_REVIEW_REQUIRED
1,2,respond,polite,0.8,AUTO_APPROVED
2,3,review,neutral,0.4,HUMAN_REVIEW_REQUIRED
3,4,respond,polite,0.8,AUTO_APPROVED
4,5,respond,polite,0.8,AUTO_APPROVED


In [7]:
def human_feedback(row):
    if row["decision"] == "HUMAN_REVIEW_REQUIRED":
        return "respond"
    return row["predicted_intent"]


In [8]:
pred_df["final_action"] = pred_df.apply(human_feedback, axis=1)
pred_df.head()


,id,predicted_intent,predicted_tone,confidence,decision,final_action
0,1,review,neutral,0.4,HUMAN_REVIEW_REQUIRED,respond
1,2,respond,polite,0.8,AUTO_APPROVED,respond
2,3,review,neutral,0.4,HUMAN_REVIEW_REQUIRED,respond
3,4,respond,polite,0.8,AUTO_APPROVED,respond
4,5,respond,polite,0.8,AUTO_APPROVED,respond


In [9]:
output_path = "../data/milestone3_output_Nikhitha.csv"
pred_df.to_csv(output_path, index=False)
print("Saved:", output_path)


Saved: ../data/milestone3_output_Nikhitha.csv


In [2]:
import pandas as pd

df = pd.read_csv("../data/sample_emails_with_triage_200.csv")
df.head()


,id,sender,subject,body,priority,triage_label
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond


 Define email assistant (creates logic)

In [3]:
def email_assistant(email_text):
    text = str(email_text).lower()

    if any(w in text for w in ["urgent", "asap", "deadline", "submit", "eod"]):
        return "notify", "urgent", 0.9
    elif any(w in text for w in ["please", "can you", "review", "confirm"]):
        return "respond", "polite", 0.8
    elif any(w in text for w in ["thank you", "thanks"]):
        return "ignore", "polite", 0.85
    else:
        return "review", "neutral", 0.4


In [4]:
results = []

for _, row in df.iterrows():
    intent, tone, confidence = email_assistant(row["body"])
    results.append({
        "id": row["id"],
        "predicted_intent": intent,
        "predicted_tone": tone,
        "confidence": confidence
    })

pred_df = pd.DataFrame(results)
pred_df.head()


,id,predicted_intent,predicted_tone,confidence
0,1,review,neutral,0.4
1,2,respond,polite,0.8
2,3,review,neutral,0.4
3,4,respond,polite,0.8
4,5,respond,polite,0.8


HITL decision logic

In [5]:
def human_in_loop(row):
    if row["confidence"] < 0.7:
        return "HUMAN_REVIEW_REQUIRED"
    else:
        return "AUTO_APPROVED"

pred_df["decision"] = pred_df.apply(human_in_loop, axis=1)


Final action after HITL

In [6]:
def human_feedback(row):
    if row["decision"] == "HUMAN_REVIEW_REQUIRED":
        return "respond"
    return row["predicted_intent"]

pred_df["final_action"] = pred_df.apply(human_feedback, axis=1)


In [7]:
final_df = pred_df.merge(
    df[["id", "body"]],
    on="id",
    how="left"
)


In [8]:
pred_df.head()


,id,predicted_intent,predicted_tone,confidence,decision,final_action
0,1,review,neutral,0.4,HUMAN_REVIEW_REQUIRED,respond
1,2,respond,polite,0.8,AUTO_APPROVED,respond
2,3,review,neutral,0.4,HUMAN_REVIEW_REQUIRED,respond
3,4,respond,polite,0.8,AUTO_APPROVED,respond
4,5,respond,polite,0.8,AUTO_APPROVED,respond


Add HITL decision 

In [9]:
def human_in_loop(row):
    if row["confidence"] < 0.7:
        return "HUMAN_REVIEW_REQUIRED"
    else:
        return "AUTO_APPROVED"

pred_df["decision"] = pred_df.apply(human_in_loop, axis=1)
pred_df.head()


,id,predicted_intent,predicted_tone,confidence,decision,final_action
0,1,review,neutral,0.4,HUMAN_REVIEW_REQUIRED,respond
1,2,respond,polite,0.8,AUTO_APPROVED,respond
2,3,review,neutral,0.4,HUMAN_REVIEW_REQUIRED,respond
3,4,respond,polite,0.8,AUTO_APPROVED,respond
4,5,respond,polite,0.8,AUTO_APPROVED,respond


Final action after Human Review

In [10]:
def human_feedback(row):
    if row["decision"] == "HUMAN_REVIEW_REQUIRED":
        return "respond"
    return row["predicted_intent"]

pred_df["final_action"] = pred_df.apply(human_feedback, axis=1)
pred_df.head()


,id,predicted_intent,predicted_tone,confidence,decision,final_action
0,1,review,neutral,0.4,HUMAN_REVIEW_REQUIRED,respond
1,2,respond,polite,0.8,AUTO_APPROVED,respond
2,3,review,neutral,0.4,HUMAN_REVIEW_REQUIRED,respond
3,4,respond,polite,0.8,AUTO_APPROVED,respond
4,5,respond,polite,0.8,AUTO_APPROVED,respond


MERGE EMAIL TEXT 

In [11]:
final_df = pred_df.merge(
    df[["id", "body"]],
    on="id",
    how="left"
)


In [12]:
final_df = final_df.rename(columns={
    "body": "email",
    "predicted_intent": "ai_action",
    "predicted_tone": "tone",
    "decision": "hitl_status"
})


In [13]:
final_df = final_df[
    ["email", "ai_action", "final_action", "hitl_status", "tone"]
]

final_df.head()


,email,ai_action,final_action,hitl_status,tone
0,Reminder: The client meeting is scheduled at 1...,review,respond,HUMAN_REVIEW_REQUIRED,neutral
1,Your invoice of INR 25515.09 is due on 2025-12...,respond,respond,AUTO_APPROVED,polite
2,Reminder: The client meeting is scheduled at 1...,review,respond,HUMAN_REVIEW_REQUIRED,neutral
3,"Hello team, please find the attached weekly re...",respond,respond,AUTO_APPROVED,polite
4,"Hello team, please find the attached weekly re...",respond,respond,AUTO_APPROVED,polite


In [14]:
output_path = "../data/milestone3_output_Nikhitha.csv"
final_df.to_csv(output_path, index=False)
print("Milestone 3 output saved at:", output_path)


Milestone 3 output saved at: ../data/milestone3_output_Nikhitha.csv
